In [15]:
import pandas as pd # import for data reading and manipulation
import re 
from pathlib import Path

In [16]:
MIMIC_DIR = Path("/home/emma/data/physionet.org/files/mimiciv/3.1/hosp/")
%store MIMIC_DIR

patients = pd.read_csv(MIMIC_DIR / "patients.csv") # Patient demographics
admissions = pd.read_csv(MIMIC_DIR / "admissions.csv", 
            parse_dates=['admittime', 'dischtime'] # Hospital admissions
)
print(f"Patients shape: {len(patients):,} rows")
print(f"Admissions shape: {len(admissions):,} rows")

Stored 'MIMIC_DIR' (PosixPath)
Patients shape: 364,627 rows
Admissions shape: 546,028 rows


In [17]:
Diagnoses = pd.read_csv( # extracting cancer specific patients
    MIMIC_DIR / "diagnoses_icd.csv",
    dtype = {"icd_code": str}
)
# Malignant neoplasms only (excludes benign 210-229, in-situ 230-234, uncertain
# behavior 235-238, and unspecified-nature 239 codes, which the old ranges
# "1[5-9]" and "2[1-3][0-9]" were pulling in alongside true malignant disease).
# ICD-9: 140-208 is the official "Malignant Neoplasms" chapter range.
# ICD-10: C00-C97 is malignant; C4A/C7A/C7B are alphanumeric extension codes for
# malignant Merkel cell carcinoma / neuroendocrine tumors that "C[0-9]{2}" can't
# match (the letter sits where a digit is expected), so they're added explicitly.
icd9_cancer = re.compile(r"^(14[0-9]|1[5-9][0-9]|20[0-8])")
icd10_cancer = re.compile(r"^(C[0-9]{2}|C4A|C7A|C7B)")
%store Diagnoses
cancer_matches = (
    ((Diagnoses.icd_version == 9) & (Diagnoses.icd_code.str.match(icd9_cancer))) |
    ((Diagnoses.icd_version == 10) & (Diagnoses.icd_code.str.match(icd10_cancer)))
)
%store cancer_matches

cancer_df = Diagnoses[cancer_matches].copy()
%store cancer_df

print(f"Cancer diagnoses rows: {len(cancer_df):,}")

Stored 'Diagnoses' (DataFrame)
Stored 'cancer_matches' (Series)
Stored 'cancer_df' (DataFrame)
Cancer diagnoses rows: 126,760


In [18]:
cohort_base = ( # Creating the base cohort by merging cancer diagnoses with admission and patient information
    cancer_df.merge(admissions, on=["subject_id", "hadm_id"], how = "inner")
    .merge(patients, on="subject_id", how="inner")
)

cohort = (cohort_base # Grouping by subject and admission to get unique admissions and patients
          .groupby(["subject_id", "hadm_id"])
          .agg(icd_codes = ("icd_code", lambda x: sorted(set(x))),
               admittime = ("admittime", "first"),
               dischtime = ("dischtime", "first"),
               deathtime = ("deathtime", "first"),
               hospital_expire_flag = ("hospital_expire_flag", "first"),
               dod = ("dod", "first"),
               anchor_age = ("anchor_age", "first"),
               anchor_year = ("anchor_year", "first"),
               gender=("gender", "first"),
               )
               .reset_index()
)

%store cohort

print(f"Unique cancer admissions: {len(cohort):,}")
print(f"Unique cancer patients: {cohort['subject_id'].nunique():,}")

cohort.to_parquet("cancer_cohort.parquet")


Stored 'cohort' (DataFrame)
Unique cancer admissions: 76,448
Unique cancer patients: 29,772


In [19]:
cohort_subject_ids = set(cohort.subject_id.unique()) # set of subjects for filtering prescriptions

opioid_pattern = re.compile( # filter for specific opioids of interest
    r"(morphine|fentanyl|oxycodone|hydrocodone|hydromorphone|tramadol|codeine|methadone|buprenorphine)",
    re.IGNORECASE
)

chunks = []
chunk_size = 1_000_000 

reader = pd.read_csv(MIMIC_DIR / "prescriptions.csv", chunksize=chunk_size, # filtering prescriptions
                     dtype = {"subject_id": "int64", "hadm_id": "Int64"},
                     parse_dates=["starttime", "stoptime"],
                     usecols = ["subject_id", "hadm_id", "drug", "dose_val_rx", 
                                "dose_unit_rx", "route", "starttime", "stoptime"],
                    low_memory=False,
)

for i, chunk in enumerate(reader): # filtering for opioid prescriptions and cohort subjects
    chunk = chunk[chunk.subject_id.isin(cohort_subject_ids)]
    chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]
    if len(chunk):
        chunks.append(chunk)
    if i % 10 == 0:
        print(f"Processed {i} chunks, opioid rows: {sum(len(c) for c in chunks):,}")

opioids = pd.concat(chunks, ignore_index=True)

print(f"Total opioid rows: {len(opioids):,}")

opioids.to_parquet("opioid_orders.parquet")

/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]


Processed 0 chunks, opioid rows: 15,437


/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]
/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]
/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]
/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.cont

Processed 10 chunks, opioid rows: 176,559


/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]
/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]
/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(opioid_pattern, case=False, na=False, regex=True)]
/tmp/ipykernel_1299314/983370121.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.cont

Processed 20 chunks, opioid rows: 323,455
Total opioid rows: 323,455


In [20]:
cohort_with_opioids = cohort.merge( # merging cohort of cancer patients with opioid prescriptions
    opioids, on = ["subject_id", "hadm_id"], how = "left"
)
cohort_with_opioids.to_parquet("cohort_with_opioids.parquet") # saving merged cohort
print("Done. Saved cancer_cohort.parquet, opioid_orders.parquet, and cohort_with_opioids.parquet")

Done. Saved cancer_cohort.parquet, opioid_orders.parquet, and cohort_with_opioids.parquet


In [21]:
exposed_hadm_ids = set(cohort_with_opioids.dropna(subset=["drug"]).hadm_id.unique()) 

n_exposed = len(exposed_hadm_ids) # num of admissions with opioid prescriptions
n_total = cohort.hadm_id.nunique() # num of total admissions in cohort
n_not_exposed = n_total - n_exposed # num of admissions without opioid prescriptions

print(f"Admissions with greater than 1 opioid_order: {n_exposed:,} ({n_exposed/n_total:.1%})")
print(f"Admissions without opioid orders: {n_not_exposed:,} ({n_not_exposed/n_total:.1%})")


Admissions with greater than 1 opioid_order: 53,202 (69.6%)
Admissions without opioid orders: 23,246 (30.4%)


In [22]:
n_died_hospital = cohort.hospital_expire_flag.sum() # num of patients who died in hospital
n_died_any = cohort.dod.notna().sum() # num of patients who died 

print(f"In-hospital deaths: {n_died_hospital:,} ({n_died_hospital/len(cohort):.1%})")
print(f"Deaths (any cause): {n_died_any:,} ({n_died_any/cohort.subject_id.nunique():.1%})")

In-hospital deaths: 3,481 (4.6%)
Deaths (any cause): 43,840 (147.3%)


In [23]:
patient_level = cohort.drop_duplicates(subset=["subject_id"]) # filters out duplicates to make cohort
death_rate = patient_level.dod.notna().mean()
print(f"Patient mortality (any recorded): {death_rate:.2%}")

Patient mortality (any recorded): 47.88%


In [24]:
funnel = { # cohort dimensions
    "Total admissions in diagnoses_cd": Diagnoses.hadm_id.nunique(),
    "Admissions with cancer ICD code": cancer_df.hadm_id.nunique(),
    "After merge with admissions/patients tables": cohort_base.hadm_id.nunique(),
    "Final cohort (unique admissions)": cohort.hadm_id.nunique(),
    "Final cohort (unique patients)": cohort.subject_id.nunique(),
    "with opioid exposure": n_exposed,
    "without opioid exposure": n_not_exposed,
    "in-hospital deaths": int(n_died_hospital),
    
}
for i, n in funnel.items():
    print(f"{i}: {n:,}")

Total admissions in diagnoses_cd: 545,497
Admissions with cancer ICD code: 76,448
After merge with admissions/patients tables: 76,448
Final cohort (unique admissions): 76,448
Final cohort (unique patients): 29,772
with opioid exposure: 53,202
without opioid exposure: 23,246
in-hospital deaths: 3,481


In [25]:
cohort["exposed"] = cohort.hadm_id.isin(exposed_hadm_ids) 
cross_tabulation = pd.crosstab( 
    cohort["exposed"],
    cohort["hospital_expire_flag"],
    margins = True 
)
cross_tabulation.columns = ["Survived", "Died", "Total"] # table for cohort
cross_tabulation.index = ["Not Exposed", "Exposed", "Total"]
print(cross_tabulation)

             Survived  Died  Total
Not Exposed     23158    88  23246
Exposed         49809  3393  53202
Total           72967  3481  76448


In [26]:
# FEATURE EXTRACTION
demo = cohort[["subject_id", "hadm_id", "anchor_age", "anchor_year", "gender"]].copy() # start of feature extraction
cohort["admit_year"] = cohort.admittime.dt.year
demo["age_at_admission"] = cohort["anchor_age"] + cohort["admit_year"] - cohort["anchor_year"]

%store demo
%store cohort


Stored 'demo' (DataFrame)
Stored 'cohort' (DataFrame)


In [27]:
#cohort = cohort.drop(columns=["admittime", "dischtime", "deathtime", 
 #           "hospital_expire_flag", "dod", "anchor_age", "anchor_year", "gender"])
#%store cohort

In [28]:
admissions = pd.read_csv(MIMIC_DIR / "admissions.csv", usecols= ["hadm_id", "insurance", "race", "language"])
%store admissions

Stored 'admissions' (DataFrame)
